# nb_finqa_bronze_to_silver

Loads raw FinQA JSON from `Files/bronze/finqa/<commit>/` into Delta tables in the
`bronze` and `silver` schemas (D14).

- Attach the **lh_finsight** lakehouse (created with schemas enabled) as the default lakehouse.
- Attach the **env_finsight** Environment (it installs the `finsight` wheel).
- Run from the `pl_finqa_ingest` pipeline, which passes the parameters below.

In [ ]:
# Parameters (mark this cell as the parameter cell in Fabric)
source_commit = "0f16e2867befa6840783e58be38c9efb9229d742"
run_id = "manual"


In [ ]:
import json
from pyspark.sql import functions as F
from finsight.spark import finqa_silver as fs
from finsight.spark.delta_io import merge_into

## Schemas
One schema per layer (D14). Idempotent, so reruns are safe. If `CREATE SCHEMA`
fails, create the schemas in the Lakehouse explorer instead (see `docs/fabric-setup.md`).

In [ ]:
for schema in ("bronze", "silver", "gold"):
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql("SHOW SCHEMAS").show()

In [ ]:
bronze_path = f"Files/bronze/finqa/{source_commit}/"
bronze = fs.read_bronze(spark, bronze_path, source_commit, run_id).cache()
print(bronze.groupBy("split").count().collect())

## Bronze table
Raw records as loaded, plus lineage columns. Append-only: every run is kept.

In [ ]:
(bronze.write.format("delta").mode("append")
   .saveAsTable("bronze.finqa_records"))

## Silver
Check quality **before** writing, so a bad load never reaches silver.

In [ ]:
docs = fs.documents(bronze)
units = fs.evidence_units(bronze)
qs = fs.questions(bronze).cache()

metrics = fs.quality_checks(bronze, docs, qs)  # raises -> notebook fails -> pipeline fails
metrics

In [ ]:
results = {
    "silver.finqa_documents": merge_into(spark, docs, "silver.finqa_documents", ["doc_id"]),
    "silver.finqa_evidence_units": merge_into(spark, units, "silver.finqa_evidence_units", ["unit_id"]),
    "silver.finqa_questions": merge_into(spark, qs, "silver.finqa_questions", ["question_id"]),
}
results

## Sanity check through Spark SQL
The same tables are also queryable from the lakehouse's SQL analytics endpoint.

In [ ]:
spark.sql("""
  SELECT split, label_status, COUNT(*) AS n
  FROM silver.finqa_questions GROUP BY split, label_status ORDER BY split, n DESC
""").show()

In [ ]:
# Return metrics to the pipeline (readable as the activity's exitValue)
notebookutils.notebook.exit(json.dumps({"run_id": run_id, "writes": results, **metrics}))